In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from embedding_metrics import evaluate_embedding_method


def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time


path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv",
}


In [ ]:
np.random.seed(42)

# ------------------------------------------
# simulate noisy data (your block unchanged)
# ------------------------------------------
simulation_results = {}
noise, extra_dim = 0.2, 6
for name, path in path_map.items():
    X_gt, V_gt, time = load_vector_field(path)
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)
    X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
    V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))
    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])
    simulation_results[name] = dict(X=X, V=V, X_gt=X_gt, V_gt=V_gt, true_time=time)

In [ ]:
from flowmap import VectorFieldEmbedder

embedding_results = {}

for name, data in simulation_results.items():
    X = data["X"]
    V = data["V"]
    time = data["true_time"]
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # normalize

    emb = VectorFieldEmbedder(
        X, V, use_PCA=False,
        embed_kwargs={"n_neighbors": 20, "min_dist": 0.4}
    )
    emb.initialize_embedding()
    # emb.optimize()

    embedding_results[name] = {
        "embedder": emb,
        "time": time
    }

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

fig, axs = plt.subplots(1, 8, figsize=(32, 4))
scores = []

for ax, name in zip(axs, embedding_results.keys()):
    emb = embedding_results[name]["embedder"]
    time = embedding_results[name]["time"]
    X_gt = simulation_results[name]["X_gt"]
    V_gt = simulation_results[name]["V_gt"]
    X_emb = emb.X_emb
    V_emb = emb.tps_vf.predict(X_emb)

    # Custom panel layout
    if name == "straight_line":
        stream_density, aspect = 0.4, 2.5
    elif name == "sine_curve":
        stream_density, aspect = 0.6, 2.0
    elif name == "branch_2":
        stream_density, aspect = 0.6, 1.5
    elif name == "branch_4":
        stream_density, aspect = 0.8, 1.5
    else:
        stream_density, aspect = 0.6, "equal"
    ax.scatter(X_emb[:, 0], X_emb[:, 1], c=time, cmap="viridis", s=30, alpha=0.5, linewidths=0)
    ax.quiver(X_emb[:, 0], X_emb[:, 1], V_emb[:, 0], V_emb[:, 1], angles="xy", scale_units="xy", scale=30, width=0.003, color="black", alpha=0.4)
    ax.set_aspect(aspect)
    ax.axis("off")

    # Score
    metrics = evaluate_embedding_method(X_gt, X_emb, V_gt, V_emb, k=30)
    metrics["dataset"] = name
    scores.append(metrics)

plt.tight_layout()
fig_dir = "./figures/simulation"
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, "flowmap_embedding_streams.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved figure to {fig_path}")
plt.show()

# Save scores
df = pd.DataFrame(scores).set_index("dataset")
os.makedirs("./data/8_vf_collection", exist_ok=True)
df.to_csv("./data/8_vf_collection/flowmap.csv")

print("\nFlowMap scores saved to ./data/8_vf_collection/flowmap.csv")
print(df.round(4))